# Reproduksi Hasil ~71% (Val Macro F1)
Notebook ini mendokumentasikan langkah reproduksi IndoBERT + BiLSTM dengan target validasi sekitar 0.71 macro F1 pada setup GPU 4GB.

## 1) Persiapan Data (CPU)
Gunakan dataset relabel dan siapkan split train/test + train_sub/val_sub + balancing train_sub.

In [ ]:
!python3 src/05_split_data.py --input data/dataset_relabel_mbg.csv --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub:', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  :', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())

In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-mode fixed --target-count 1500 --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json

## 2) Training IndoBERT + BiLSTM (CUDA)
Gunakan config yang sudah terbukti stabil pada GPU 4GB.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
# os.environ['HF_TOKEN'] = 'hf_xxx'  # optional

In [ ]:
!python3 src/07_indobert_bilstm.py --train data/train_sub_balanced.csv --val data/val_sub.csv --trial-configs-json src/resources/step7_cuda_4gb_ultrasafe.json --max-trials 2

## 3) Evaluasi Final di Test Set Asli
Evaluasi harus tetap di `data/test.csv` yang tidak dibalance.

In [ ]:
!python3 src/09_evaluate.py --test data/test.csv --model-path models/best_indobert_bilstm.pt

In [ ]:
import json, pandas as pd
m = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
print(m)
display(pd.read_csv('outputs/classification_report.csv'))